# VaultGuard — Exploratory Data Analysis


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd().resolve()

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found: {DATA_PATH}. Download creditcard.csv and place it in data/raw/.'
    )

sns.set_theme(style='whitegrid')
df = pd.read_csv(DATA_PATH)
print(f'Dataset path: {DATA_PATH}')
print(f'Shape: {df.shape}')
display(df.head())
df.info()
display(df.describe().T)

## Data quality and target balance

The fraud class is expected to be extremely rare. Accuracy alone is therefore not a reliable success metric for the later models.

In [ ]:
expected_columns = {'Time', 'Amount', 'Class', *[f'V{i}' for i in range(1, 29)]}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Dataset is missing expected columns: {sorted(missing_columns)}')

print('Missing values by column:')
display(df.isnull().sum().to_frame('missing_values').T)
print(f'Duplicate rows: {df.duplicated().sum():,}')

class_counts = df['Class'].value_counts().sort_index()
class_percentages = (df['Class'].value_counts(normalize=True).sort_index() * 100)
class_summary = pd.DataFrame({'count': class_counts, 'percentage': class_percentages})
class_summary.index = class_summary.index.map({0: 'Legitimate', 1: 'Fraud'})
display(class_summary)

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='Class', hue='Class', palette=['#4C78A8', '#E45756'], legend=False)
ax.set_title('Legitimate vs Fraudulent Transactions')
ax.set_xlabel('Transaction class')
ax.set_ylabel('Transactions')
ax.set_xticks([0, 1], ['Legitimate', 'Fraud'])
plt.tight_layout()
plt.show()

## Transaction amount and time

The `V1`–`V28` fields are anonymized PCA-transformed variables. This analysis does not claim they correspond to specific banking attributes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df, x='Amount', hue='Class', bins=100, log_scale=(False, True), element='step', stat='count', common_norm=False, ax=axes[0])
axes[0].set_title('Transaction Amount Distribution')

sns.boxplot(data=df, x='Class', y='Amount', hue='Class', palette=['#4C78A8', '#E45756'], legend=False, ax=axes[1])
axes[1].set_ylim(0, 2500)
axes[1].set_xticks([0, 1], ['Legitimate', 'Fraud'])
axes[1].set_title('Transaction Amount by Class (capped at 2,500)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.histplot(data=df, x='Time', hue='Class', bins=100, element='step', stat='count', common_norm=False)
plt.title('Fraud Distribution Over Time')
plt.xlabel('Time')
plt.tight_layout()
plt.show()

## EDA conclusions

- The data is heavily imbalanced, with fraud making up a very small share of transactions.
- Future models should be assessed with precision, recall, F1, ROC-AUC, and especially PR-AUC—not accuracy alone.
- Transaction amount and time can be inspected for distributional differences, but the anonymized `V` features must not be assigned speculative real-world meanings.
- The next stage will use a stratified train/test split so both partitions retain the rare fraud proportion.